![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 47 -- Lab 1: Exam Forecast

Every summer the students in their last year of high school take a national exam, and every school gets an average grade in every subject. You write the program that forecasts next year's averages from the years before. Your file is scored with MAE, the mean distance between your numbers and the real averages: lower is better.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The exam results of about 1,700 schools from 2014 to 2021, one row per school, subject and year, with the average grade (`average`). The test rows are the year 2022, without the average; only the scorer holds it. The setup cell leaves in `DATA_DIR`: `train.csv`, `test.csv` (the same columns without `average`) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="tab-01", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("libraries ready")


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/exam-forecast")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The task

**The story.** Every summer, the students in their last year of high school take a national exam. Over the years
its results give a detailed picture of every school: some subjects go up, some go down, some schools do well year
after year. The exam office wants to forecast next year's results from the years before.

**One data point** is one school, one subject and one year: how that school's candidates did in that subject that
year. The columns:

| column | what it is |
|---|---|
| `id` | the number of the row |
| `year` | the exam year: 2014 to 2021 in `train.csv`, 2022 in `test.csv` |
| `school` | the school, as a code such as `S0002` |
| `county` | the county the school is in, as a code such as `C06` |
| `subject` | the exam subject; `OVERALL` is the row of the school's overall results that year (all its candidates) |
| `candidates` | how many of the school's students took that subject that year |
| `pass_rate` | the percentage of those candidates who passed |
| `subject_share` | the percentage of the school's candidates who chose that subject |
| `average` | only in `train.csv`: the mean grade of those candidates, from 1 to 10. **This is the answer.** |

**The files.** The first two rows of `train.csv`:

| id | year | school | county | subject | candidates | pass_rate | subject_share | average |
|---|---|---|---|---|---|---|---|---|
| 0 | 2014 | S0002 | C06 | Computer science C++ (math-CS track) | 56 | 100.0 | 19.5 | 9.26 |
| 1 | 2014 | S0007 | C35 | Computer science C++ (math-CS track) | 12 | 100.0 | 7.1 | 8.93 |

Row 0 reads: in 2014, 56 students of school S0002 took Computer science C++, all of them passed, they were 19.5% of
the school's candidates, and their mean grade was 9.26. The first two rows of `test.csv`, the rows you answer for:

| id | year | school | county | subject | candidates | pass_rate | subject_share |
|---|---|---|---|---|---|---|---|
| 107062 | 2022 | S0002 | C06 | Computer science C++ (math-CS track) | 61 | 100.0 | 22.8 |
| 107063 | 2022 | S0007 | C35 | Computer science C++ (math-CS track) | 9 | 100.0 | 5.2 |

**Your job.** Forecast `average` for every row of `test.csv`.

**The organisers' hint.** Any method that works from the data of the earlier years is allowed. The organisers
suggest four things to take into account: the subject, the school's own past results, how results change over the
years, and how the county or the whole country did.

**The file you hand in**, `submission.csv`, has one row per test row, in the format of `sample_submission.csv`:

| id | average |
|---|---|
| 107062 | 5.0 |
| 107063 | 5.0 |

**The score** is MAE (mean absolute error) on the test rows, computed by the scorer cell in Part 2. Lower is
better; 0 would be perfect.

**The rules.** Any library; no outside data. Your notebook must run from start to end.

**The starter** (next cell) gives every test row a random average between 1 and 10. The score of its file is the
**baseline**; your points count from there.

**Today you work alone.** Nobody helps with the code and AI assistants are off. This morning's slides are yours
to use.


---
# The starter solution

Every olympiad task comes with a starter: something that hands in a valid file. Its score is the **baseline**, and your points count up from it. Below is this task's own starter, with English names, cut to the question of this lab. Read it before you begin; you may reuse any of its lines.

```python
import numpy as np
import pandas as pd

np.random.seed(42)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# Every test row gets a random average between 1 and 10
answers = []
for _, row in test.iterrows():
    answers.append(np.round(np.random.uniform(1, 10), 2))

submission = pd.DataFrame({"id": test["id"], "average": answers})
submission.to_csv("submission.csv", index=False)
print(f"Submission file saved: {len(submission)} rows")
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the data before you build anything.

## Task 1: Load the files

Read the three CSV files of `DATA_DIR` into `train`, `test` and `sample`. Print the shape of each.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample)


## Task 2: Look at the training rows

Store the first five rows of `train` in `first_rows` and its number of rows in `n_rows`. Print `first_rows`, then the smallest, the mean and the largest `average` of `train`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task. Run the cell; you will use `score` all afternoon.

In [ ]:
from sklearn.metrics import mean_absolute_error


def score(answers, numbers):
    """The score of a file: the mean absolute error (MAE) of its numbers against the answers. Lower is better."""
    return float(mean_absolute_error(answers, numbers))


print(round(score([6.0, 8.0, 9.5], [7.0, 8.0, 7.5]), 2))


## Task 3: What the scorer gives

The scorer cell above prints 1.0 for three rows. Which sentence says what `score` gives?

- **a** The mean of the squared misses, (1 x 1 + 0 + 2 x 2) / 3 = 1.67: a big miss counts more than its size.
- **b** The mean distance between each number and its answer, (1 + 0 + 2) / 3 = 1.0: a miss counts as it is.
- **c** The number of rows that the file misses by more than half a grade: one row of the three, so 1.0.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two files that need no model score? `five` gives every row the number 5.0, the number of the sample file; `median` gives every row the median `average` of `train`. Score both on the rows of `train` against their `average` with `score`, and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
# Your code here


## Task 5: What the metric rewards

File A misses every row by 0.5. File B is exactly right on nine rows in ten, but misses the tenth by 5. Which file has the lower (better) score?

- **a** File A: the one miss of 5 costs 5 x 5 = 25, far more than all the small misses of 0.5 together.
- **b** File B, because it is exactly right on nine rows in ten, while file A is right on none of them.
- **c** Neither: a miss counts as it is, so both files score 0.5. One miss of 5 costs what ten of 0.5 cost.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, `id_col`, the name of its id column, and `pred_col`, the name of the column of your numbers. Read all three from what you loaded; do not type them.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: aim to hand it in within the first hour, then make it better.

## Task 8: Hold out validation rows

Split `train` into a part `tr` that your model learns from and a validation part `va` that you score on. Keep the ids of the two parts in `train_ids` and `val_ids`. Print the number of rows in each part.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


## Task 9: The numbers the model reads

Build `X_tr`, the table of numbers your model reads for the rows of `tr` (one row per row of `tr`), and `y_tr`, their `average`. Build `X_va` and `y_va` the same way for the rows of `va`. Print the shapes.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(X_tr, y_tr)


## Task 10: A first model

Train a first model in `model` on `X_tr` and `y_tr`. It must give one number per row. Print what it gives the first three rows of `X_va`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, X_va)


## Task 11: Score the validation rows

Store in `val_score` the `score` of your model's numbers for the rows of `X_va` against `y_va`: one number. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="mae")


## Task 12: Write the submission file

Write `submission.csv` in the format of `sample`: for every row of `test`, its `id` and your number. Leave the index out. Print its shape and its first rows.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Debugging drill

A classmate tried to list the five subjects with the highest mean `average` in `train`. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it prints `drill_top`, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-typ-1")


## Task 13: Repair the classmate's cell

Change as little as you can: `drill_top` stays the five largest means, as numbers.

In [ ]:
means = train.groupby("subject").mean()
drill_top = means.sort_values(ascending=False).head(5).values
print(drill_top)


In [ ]:
ck.drill_check("bug-typ-1", drill_top)


---
# Part 5 -- Challenge: your best file

Your first file is on the board. Now make it better, one change at a time. A change worth keeping comes from something you found in the data or in the rules, and it lowers the score of the validation rows.

## Task 14: Look again

Look again at the data, all of it, for something your first file does not use yet. Print what you find.

In [ ]:
# Your code here


## Task 15: Try a better way

Make at least one change to your pipeline. Score it on your validation rows with `score`, and print it next to `val_score`.

In [ ]:
# Your code here


## Task 16: Hand in your best file

Write `submission.csv` again with your best way. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
## Task 17: Reflect

Which change moved your validation score most, and how did you find it: in the data, in the rules or by trying? Which rows did you validate on, and why those? If you met this task again with one hour, what would you do in the first 15 minutes?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task first**: the scorer read as code, two files without a model, what MAE rewards, and the format of the file, all before any model.
- **A first honest file**: a simple model, scored on validation rows with the task's own metric, handed in within the first hour.
- **Forecast from the past**: the years before, looked at per school and subject, turned into numbers a model can read.
- **Keep a change only when the score says so**: each change scored on validation rows like the test's before a new file.
- **Debugging**: a type error read from its message, located and repaired against the clock.

---

Made By **Sattam Altwaim**